
# Real Data Pipeline - Data Quality Checks
## Comprehensive validation framework for GitHub data

In [0]:

# Purpose: Validate data quality at each layer



import pyspark.sql.functions as F
from datetime import datetime

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("DATA QUALITY VALIDATION FRAMEWORK")
print("=" * 70)


In [0]:

# Cell 1: Create Quality Metrics Table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.quality_metrics (
    check_id STRING,
    check_name STRING,
    table_name STRING,
    layer STRING,
    check_type STRING,
    total_records INT,
    failed_records INT,
    pass_rate DOUBLE,
    severity STRING,
    check_timestamp TIMESTAMP,
    status STRING,
    error_message STRING
)
USING DELTA
""")

print("✅ Quality metrics table created")


In [0]:

# Cell 2: Bronze Layer Quality Checks
print("\n" + "=" * 70)
print("BRONZE LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

bronze_repos = spark.table(f"{catalog}.{schema}.bronze_repositories")
bronze_contribs = spark.table(f"{catalog}.{schema}.bronze_contributors")

# Check 1: NULL values in critical fields
null_check_repos = bronze_repos.filter(
    F.col("repo_name").isNull() | 
    F.col("stars").isNull() | 
    F.col("language").isNull()
).count()

print(f"✅ Bronze Repos - NULL check: {null_check_repos} issues found")

# Check 2: Negative values (impossible for GitHub metrics)
negative_check = bronze_repos.filter(
    (F.col("stars") < 0) | 
    (F.col("forks") < 0) | 
    (F.col("watchers") < 0)
).count()

print(f"✅ Bronze Repos - Negative values check: {negative_check} issues")

# Check 3: Duplicate repositories
duplicate_check = bronze_repos.groupBy("repo_name").count().filter(F.col("count") > 1).count()

print(f"✅ Bronze Repos - Duplicates: {duplicate_check} duplicates")

# Check 4: Date validation (created_at before updated_at)
date_logic_check = bronze_repos.filter(
    F.col("created_at") > F.col("updated_at")
).count()

print(f"✅ Bronze Repos - Date logic: {date_logic_check} anomalies")

# Check 5: Contributors NULL check
contrib_null_check = bronze_contribs.filter(
    F.col("contributor_login").isNull() | 
    F.col("repo_name").isNull()
).count()

print(f"✅ Bronze Contributors - NULL check: {contrib_null_check} issues")